# Context-aware BERT – Nhận diện cảm xúc trong hội thoại (Nhóm 9)

**Yêu cầu:** Bật GPU trước khi chạy: `Runtime` → `Change runtime type` → **T4 GPU** → `Save`.

In [9]:
# 1. Clone repo và cài dependencies
!rm -rf /content/project_nhom9
!git clone https://github.com/trangdx2602/bert-context.git /content/project_nhom9
%cd /content/project_nhom9/Codebase

!pip install -r requirements.txt -q

import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("LỖI: Chưa bật GPU trên Google Colab.")

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/project_nhom9'...
fatal: Unable to read current working directory: No such file or directory
[Errno 2] No such file or directory: '/content/project_nhom9/Codebase'
/content/project_nhom9/Codebase
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.
GPU: Tesla T4


## Upload dữ liệu MELD

Chạy cell bên dưới và upload 3 file CSV:
- `train_sent_emo.csv`
- `val_sent_emo.csv`
- `test_sent_emo.csv`

In [2]:
# 2. Upload file dữ liệu MELD
import os
from google.colab import files

os.makedirs('/content/project_nhom9/Documents', exist_ok=True)

print("Upload 3 file: train_sent_emo.csv, val_sent_emo.csv, test_sent_emo.csv")
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = f'/content/project_nhom9/Documents/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f"  Đã lưu: {dest}")

Upload 3 file: train_sent_emo.csv, val_sent_emo.csv, test_sent_emo.csv


Saving test_sent_emo.csv to test_sent_emo.csv
Saving train_sent_emo.csv to train_sent_emo.csv
Saving val_sent_emo.csv to val_sent_emo.csv
  Đã lưu: /content/project_nhom9/Documents/test_sent_emo.csv
  Đã lưu: /content/project_nhom9/Documents/train_sent_emo.csv
  Đã lưu: /content/project_nhom9/Documents/val_sent_emo.csv


## Huấn luyện – Ablation context window (k = 1, 3, 5)

Chạy tuần tự 3 cell bên dưới để so sánh ảnh hưởng của độ dài context.

In [3]:
# 3a. Huấn luyện với k=1 (1 câu trước + câu hiện tại)
!python train.py --model bert_context --context_k 1 --batch_size 32 --num_workers 2


  Model            : bert_context
  Context k        : 1
  Device           : cuda
  Mixed Precision  : ON
  Batch size       : 32  (effective: 32)
  Accum steps      : 1
  Epochs           : 10

[1/4] Đang pre-tokenize và load dữ liệu...
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 220kB/s]
vocab.txt: 232kB [00:00, 2.96MB/s]
tokenizer.json: 466kB [00:00, 4.75MB/s]
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Train: 9,989 samples
  Val  : 1,109 samples

[2/4] Class weights: ['0.303', '1.184', '5.325', '2.089', '0.819', '5.266', '1.287']

[3/4] Load model...
config.json: 100% 570/570 [00:00<00:00, 2.46MB/s]
model.safetensors: 100% 440M/440M [00:02<00:00, 198MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 907.56it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight              

In [4]:
# 3b. Huấn luyện với k=3 (3 câu trước + câu hiện tại)
!python train.py --model bert_context --context_k 3 --batch_size 32 --num_workers 2


  Model            : bert_context
  Context k        : 3
  Device           : cuda
  Mixed Precision  : ON
  Batch size       : 32  (effective: 32)
  Accum steps      : 1
  Epochs           : 10

[1/4] Đang pre-tokenize và load dữ liệu...
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Train: 9,989 samples
  Val  : 1,109 samples

[2/4] Class weights: ['0.303', '1.184', '5.325', '2.089', '0.819', '5.266', '1.287']

[3/4] Load model...
Loading weights: 100% 199/199 [00:00<00:00, 1531.23it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bia

In [5]:
# 3c. Huấn luyện với k=5 (5 câu trước + câu hiện tại)
!python train.py --model bert_context --context_k 5 --batch_size 32 --num_workers 2


  Model            : bert_context
  Context k        : 5
  Device           : cuda
  Mixed Precision  : ON
  Batch size       : 32  (effective: 32)
  Accum steps      : 1
  Epochs           : 10

[1/4] Đang pre-tokenize và load dữ liệu...
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Train: 9,989 samples
  Val  : 1,109 samples

[2/4] Class weights: ['0.303', '1.184', '5.325', '2.089', '0.819', '5.266', '1.287']

[3/4] Load model...
Loading weights: 100% 199/199 [00:00<00:00, 1056.61it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bia

## Đánh giá trên tập test

In [6]:
# 4a. Đánh giá k=1
!python evaluate.py --model bert_context --context_k 1 --num_workers 2


Device: cuda  |  AMP: ON
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Đang pre-tokenize test...
Test set: 2,610 samples

Loading weights: 100% 199/199 [00:00<00:00, 1000.83it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  [Checkpoint loaded] epoch=2, val_f1=0.55

In [7]:
# 4b. Đánh giá k=3
!python evaluate.py --model bert_context --context_k 3 --num_workers 2


Device: cuda  |  AMP: ON
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Đang pre-tokenize test...
Test set: 2,610 samples

Loading weights: 100% 199/199 [00:00<00:00, 1279.72it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  [Checkpoint loaded] epoch=10, val_f1=0.5

In [8]:
# 4c. Đánh giá k=5
!python evaluate.py --model bert_context --context_k 5 --num_workers 2


Device: cuda  |  AMP: ON
  Đang pre-tokenize train...
  Đang pre-tokenize val...
  Đang pre-tokenize test...
Test set: 2,610 samples

Loading weights: 100% 199/199 [00:00<00:00, 1349.42it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  [Checkpoint loaded] epoch=8, val_f1=0.58

## Tổng hợp kết quả

Điền kết quả sau khi chạy xong vào bảng bên dưới.

| Mô hình | Context k | Accuracy | Weighted F1 |
|---------|-----------|----------|-------------|
| Context-aware BERT | k=1 | | |
| Context-aware BERT | k=3 | | |
| Context-aware BERT | k=5 | | |
